In [23]:
%pip install python-dotenv
%pip install polars


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import os
import requests
import pandas as pd
import polars as pl
import numpy as np
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise ValueError("API_KEY not found in .env")

In [25]:
url = "https://api.511.org/transit/datafeeds"

In [26]:
import zipfile

params = {
    "api_key": API_KEY,
    "operator_id": "RG",
    "historic": "2025-03-so"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()

with open("../data/march_2025_gtfs.zip", "wb") as f:
    f.write(response.content)

with zipfile.ZipFile("../data/march_2025_gtfs.zip", "r") as zip_ref:
    zip_ref.extractall("march_2025_gtfs")

os.listdir("../data/march_2025_gtfs")

['transfers.txt',
 'agency.txt',
 'calendar_dates.txt',
 'stop_times.txt',
 'levels.txt',
 'directions.txt',
 'trips.txt',
 'stop_observations.txt',
 'pathways.txt',
 'stops.txt',
 'routes.txt']

In [27]:
print(os.getcwd())
print(os.path.exists("../data"))
print(os.listdir("../data") if os.path.exists("../data") else "no data folder")
print(os.path.exists("../data/march_2025_gtfs"))
print(os.listdir("../data/march_2025_gtfs") if os.path.exists("../data/march_2025_gtfs") else "no extracted folder")

/Users/johnluke/Desktop/CYPLAN101Project/CYPLAN101_Project_G16/notebooks
True
['march_2025_gtfs', 'march_2025_gtfs.zip', 'processed']
True
['transfers.txt', 'agency.txt', 'calendar_dates.txt', 'stop_times.txt', 'levels.txt', 'directions.txt', 'trips.txt', 'stop_observations.txt', 'pathways.txt', 'stops.txt', 'routes.txt']


In [28]:
zip_path = "../data/march_2025_gtfs.zip"
print(os.path.getsize(zip_path))        # a real feed is many MB, not a few hundred bytes
print(zipfile.is_zipfile(zip_path))     # want True

430308953
True


In [29]:
with zipfile.ZipFile("../data/march_2025_gtfs.zip") as z:
    for info in z.infolist():
        print(f"{info.filename:35s} {info.file_size / 1e6:10.1f} MB")
    z.extractall("../data/march_2025_gtfs")

agency.txt                                 0.0 MB
calendar_dates.txt                       141.2 MB
directions.txt                             0.0 MB
levels.txt                                 0.0 MB
pathways.txt                               0.1 MB
routes.txt                                 0.1 MB
stop_observations.txt                   2059.7 MB
stop_times.txt                           602.4 MB
stops.txt                                  2.5 MB
transfers.txt                              0.0 MB
trips.txt                                 21.4 MB


In [30]:
obs = pd.read_csv(
    "../data/march_2025_gtfs/stop_observations.txt",
    dtype={
        "trip_id": "string",
        "vehicle_id": "string",
        "route_id": "string",
        "agency_id": "string",
        "from_stop_id": "string",
        "to_stop_id": "string"
    }
)

sf = obs[
    obs["agency_id"].str.contains("SF", na=False)
].copy()

sf.head(10)

,trip_id,trip_start_time,schedule_relationship,service_date,vehicle_id,stop_sequence,observed_arrival_time,observed_departure_time,uncertainty,dwell_time_secs,scheduled_dwell_time_secs,route_id,agency_id,direction_id,from_stop_id,to_stop_id,scheduled_arrival_time,scheduled_departure_time
416,SF:11721738_M31:20250331,NaN,0,20250331,8857,1,05:00:00,05:00:00,0,0.0,0.0,SF:29,SF,1.0,<NA>,14648,05:00:00,05:00:00
417,SF:11721738_M31:20250331,NaN,0,20250331,8857,2,05:01:01,05:01:01,0,0.0,0.0,SF:29,SF,1.0,14648,14647,05:00:33,05:00:33
418,SF:11721738_M31:20250331,NaN,0,20250331,8857,3,05:01:58,05:01:58,0,0.0,0.0,SF:29,SF,1.0,14647,14646,05:01:14,05:01:14
419,SF:11721738_M31:20250331,NaN,0,20250331,8857,4,05:02:22,05:02:22,0,0.0,0.0,SF:29,SF,1.0,14646,14645,05:01:59,05:01:59
420,SF:11721738_M31:20250331,NaN,0,20250331,8857,5,05:03:38,05:03:38,0,0.0,0.0,SF:29,SF,1.0,14645,14784,05:02:54,05:02:54
421,SF:11721738_M31:20250331,NaN,0,20250331,8857,6,05:03:57,05:03:57,0,0.0,0.0,SF:29,SF,1.0,14784,15955,05:03:27,05:03:27
422,SF:11721738_M31:20250331,NaN,0,20250331,8857,7,05:04:32,05:04:32,0,0.0,0.0,SF:29,SF,1.0,15955,15115,05:04:27,05:04:27
423,SF:11721738_M31:20250331,NaN,0,20250331,8857,8,05:05:09,05:05:09,0,0.0,0.0,SF:29,SF,1.0,15115,15116,05:05:07,05:05:07
424,SF:11721738_M31:20250331,NaN,0,20250331,8857,9,05:05:52,05:05:52,0,0.0,0.0,SF:29,SF,1.0,15116,14976,05:05:43,05:05:43
425,SF:11721738_M31:20250331,NaN,0,20250331,8857,10,05:06:25,05:06:25,0,0.0,0.0,SF:29,SF,1.0,14976,14785,05:06:23,05:06:23


In [31]:
sf.shape

(8160083, 18)

In [32]:
sf.isna().sum().sort_values(ascending=False)

trip_start_time              8160083
from_stop_id                  220269
scheduled_departure_time         319
scheduled_arrival_time           319
scheduled_dwell_time_secs        319
direction_id                      35
to_stop_id                         0
agency_id                          0
route_id                           0
trip_id                            0
uncertainty                        0
observed_departure_time            0
observed_arrival_time              0
stop_sequence                      0
vehicle_id                         0
service_date                       0
schedule_relationship              0
dwell_time_secs                    0
dtype: int64

In [33]:
sf.describe(include="all")

,trip_id,trip_start_time,schedule_relationship,service_date,vehicle_id,stop_sequence,observed_arrival_time,observed_departure_time,uncertainty,dwell_time_secs,scheduled_dwell_time_secs,route_id,agency_id,direction_id,from_stop_id,to_stop_id,scheduled_arrival_time,scheduled_departure_time
count,8160083,0,8160083.0,8.160083e+06,8160083,8.160083e+06,8160083,8160083,8160083.0,8160083.0,8159764.0,8160083,8160083,8.160048e+06,7939814,8160083,8159764,8159764
unique,62013,0,NaN,NaN,1050,NaN,96273,96273,NaN,NaN,NaN,70,1,NaN,3263,3276,79928,79928
top,SF:11733360_M31:20250314,NaN,NaN,NaN,8526,NaN,06:00:00,06:00:00,NaN,NaN,NaN,SF:29,SF,NaN,14725,15926,07:00:00,07:00:00
freq,1207,NaN,NaN,NaN,19027,NaN,779,779,NaN,NaN,NaN,392786,8160083,NaN,11356,13552,2636,2636
mean,NaN,NaN,0.0,2.025032e+07,NaN,2.356641e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,5.059219e-01,NaN,NaN,NaN,NaN
std,NaN,NaN,0.0,7.754517e+00,NaN,1.732639e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,4.999650e-01,NaN,NaN,NaN,NaN
min,NaN,NaN,0.0,2.025030e+07,NaN,1.000000e+00,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN
25%,NaN,NaN,0.0,2.025031e+07,NaN,1.000000e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN
50%,NaN,NaN,0.0,2.025032e+07,NaN,2.000000e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,1.000000e+00,NaN,NaN,NaN,NaN
75%,NaN,NaN,0.0,2.025032e+07,NaN,3.300000e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,1.000000e+00,NaN,NaN,NaN,NaN


In [34]:
sf["route_id"].nunique()

70

In [35]:
df = sf.copy()

df["service_date"] = pd.to_datetime(df["service_date"], format="%Y%m%d")
df["stop_sequence"] = pd.to_numeric(df["stop_sequence"])

# Filter for March 2025 weekdays only
df = df[(df["service_date"].dt.month == 3) & (df["service_date"].dt.dayofweek < 5)]

# trip_start_time is always NaN
df = df.drop(columns=["trip_start_time"], errors="ignore")

# one row per trip-day-stop
key = ["service_date", "trip_id", "stop_sequence"]
print("duplicate keys:", df.duplicated(key).sum())
df = df.drop_duplicates(key).sort_values(key).reset_index(drop=True)

# check what these columns contain before deciding whether to filter on them
print(df["schedule_relationship"].value_counts(dropna=False))
print(df["uncertainty"].value_counts(dropna=False))

duplicate keys: 41099
schedule_relationship
0    5803743
Name: count, dtype: int64
uncertainty
0    5803743
Name: count, dtype: int64


In [36]:
#adding new columns to convert the times to specific econds within a day, to easily subtract times
def to_sec(s):
    p = s.str.split(":", expand=True)
    return (pd.to_numeric(p[0]) * 3600
            + pd.to_numeric(p[1]) * 60
            + pd.to_numeric(p[2]))

for c in ["observed_arrival_time", "observed_departure_time",
          "scheduled_arrival_time", "scheduled_departure_time"]:
    df[c + "_s"] = to_sec(df[c])

print(df[[c for c in df.columns if c.endswith("_s")]].isna().sum())

observed_arrival_time_s         0
observed_departure_time_s       0
scheduled_arrival_time_s      319
scheduled_departure_time_s    319
dtype: int64


In [37]:
#calculates time between stops and any delays

g = df.groupby(["service_date", "trip_id"])

prev_obs_dep   = g["observed_departure_time_s"].shift(1)
prev_sched_dep = g["scheduled_departure_time_s"].shift(1)
prev_to_stop   = g["to_stop_id"].shift(1)

df["obs_seg_s"]   = df["observed_arrival_time_s"]  - prev_obs_dep
df["sched_seg_s"] = df["scheduled_arrival_time_s"] - prev_sched_dep
df["delay_s"]     = df["obs_seg_s"] - df["sched_seg_s"]

# validation: the previous row's to_stop should equal this row's from_stop
mismatch = df["from_stop_id"].notna() & (df["from_stop_id"] != prev_to_stop)
print("stop-chain mismatches:", mismatch.sum())

# first stops have no segment + any rows with a missing arrival time/previous departure time, can't use the data
seg = df.dropna(subset=["from_stop_id", "obs_seg_s", "sched_seg_s"]).copy()

stop-chain mismatches: 2


In [20]:
# defining the different times outlined, AM peak, midday, PM peak, evening, other includes all overnight trips between 10PM and 6AM
hr = (prev_sched_dep.reindex(seg.index) // 3600)
seg["period"] = np.select(
    [(hr >= 6) & (hr < 9), (hr >= 9) & (hr < 15),
     (hr >= 15) & (hr < 19), (hr >= 19) & (hr < 22)],
    ["AM peak", "midday", "PM peak", "evening"],
    default="other",
)
print(seg["period"].value_counts())
seg.to_csv("../data/processed/sf_segments_clean.csv", index=False)

period
midday     1893605
PM peak    1292788
AM peak     959369
evening     752276
other       748014
Name: count, dtype: int64


In [22]:
n0 = len(seg)

# drops the 2 stop-chain mismatches
bad = mismatch[mismatch].index.intersection(seg.index)
seg = seg.drop(index=bad)

# drops impossible and extreme values, 
# including early busses, and delays beyond 30 minutes to 
# remove any outliers, which is < 1% of the data
seg = seg[(seg["obs_seg_s"] > 0) & (seg["sched_seg_s"] > 0)
          & (seg["delay_s"].abs() <= 1800)]

# drops the overnight rides
seg = seg[seg["period"] != "other"].reset_index(drop=True)

print(f"removed {n0 - len(seg):,} of {n0:,} rows")
print(seg["period"].value_counts())
print(seg["delay_s"].describe())

seg.to_csv("../data/processed/sf_segments_clean.csv", index=False)

removed 825,362 of 5,646,052 rows
period
midday     1863512
PM peak    1272828
AM peak     944799
evening     739551
Name: count, dtype: int64
count    4.820690e+06
mean     5.807142e+00
std      6.572370e+01
min     -1.355000e+03
25%     -1.400000e+01
50%      0.000000e+00
75%      1.700000e+01
max      1.800000e+03
Name: delay_s, dtype: float64
